# Fase 3: Tratamiento Numérico, Optimización con Validación Cruzada y Entrenamiento del Modelo Híbrido
## Premier League Match Recommender

Este notebook documenta la **Fase 3 del proyecto**, donde se integran la analítica numérica objetiva y el modelado predictivo avanzado:
1. **Tratamiento e Ingeniería de Variables Numéricas y Tácticas**:
   - Goles Esperados Combinados ($xG_{comb}$).
   - Paridad competitiva en la clasificación.
   - Disponibilidad agregada de estrellas (*Star Power* ICT Index).
   - Modelado del Choque Táctico Operativo (pases progresivos, presiones defensivas, arquetipos y similitud de cosenos).
2. **Entrenamiento 1: Calibración y Optimización de Pesos Canónicos por Validación Cruzada ($k$-Fold CV)**:
   - Búsqueda en simplex regularizada sobre benchmark histórico con $k=5$ pliegues.
   - Evaluación de métricas de ranking: Correlación de Spearman, NDCG@3, MSE y $R^2$.
   - Obtención del perfil calibrado óptimo (`cv_optimal`).
3. **Entrenamiento 2: Modelo Supervisado de Machine Learning Multimodal (`GradientBoostingRegressor`)**:
   - Ensamble de 12 variables multimodales continuas.
   - Entrenamiento del regresor por gradiente potenciado e interpretabilidad de *Feature Importances*.
   - Descomposición de predicciones en macro-dimensiones de espectáculo deportivo.

In [1]:
import sys
import os
from pathlib import Path
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from sklearn.model_selection import KFold
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error, r2_score

BASE_DIR = Path(os.path.abspath("..")) if Path("..").resolve().name == "sistema_de_recomendaciones" else Path(os.path.abspath("."))
if str(BASE_DIR) not in sys.path:
    sys.path.insert(0, str(BASE_DIR))

from src.core.config import settings
from src.core.schemas import MatchNumericalFeatures, MatchScoreWeights
from src.data_ingestion.local_provider import LocalDataProvider
from src.features.numerical import NumericalFeatureExtractor
from src.features.nlp import NLPFeatureExtractor
from src.engine.hybrid_model import CrossValidationWeightOptimizer, HybridMLScorer, compute_ndcg_at_k

print("✔ Dependencias de Machine Learning y optimización importadas exitosamente.")

✔ Dependencias de Machine Learning y optimización importadas exitosamente.

---
## 1. Tratamiento e Ingeniería de Variables Numéricas y Tácticas

Calculamos y normalizamos las variables estadísticas de cada club para la jornada 29:
- **Combined Expected Goals ($xG_{comb}$)**: $xG_L + xG_V$
- **Índice de Paridad en la Tabla**: $1 - \frac{|Pos_L - Pos_V|}{20}$
- **Star Power**: $ICT_L + ICT_V$
- **Choque Táctico Operativo**: Estimación de pases progresivos (PP) y presiones defensivas (DP), clasificación de arquetipos tácticos y cálculo de la intensidad del choque $[0.10, 1.00]$.

In [2]:
local_provider = LocalDataProvider(data_dir=settings.data_dir)
num_extractor = NumericalFeatureExtractor()
fixtures = local_provider.get_fixtures(gameweek=29)

numerical_records = []
for f in fixtures:
    num_feat = num_extractor.extract(f)
    numerical_records.append({
        "fixture_id": f.id,
        "match": f"{f.home_team.name} vs {f.away_team.name}",
        "xg_combined": num_feat.xg_combined,
        "table_parity": num_feat.table_parity,
        "star_power": num_feat.star_power,
        "tactical_label": num_feat.tactical_style_label,
        "tactical_intensity": num_feat.tactical_clash_intensity,
        "prog_passes": num_feat.progressive_passes_combined,
        "pressures": num_feat.defensive_pressures_combined,
        "normalized_num_score": num_feat.normalized_numerical_score
    })

df_numerical = pd.DataFrame(numerical_records).sort_values("normalized_num_score", ascending=False).reset_index(drop=True)
print("--- VARIABLES NUMÉRICAS Y TÁCTICAS (JORNADA 29) ---")
print(df_numerical[["match", "xg_combined", "table_parity", "star_power", "tactical_intensity", "normalized_num_score"]].to_string(index=False))

print("\n--- DETALLE DE CHOQUE TÁCTICO OPERATIVO ---")
for _, r in df_numerical[["match", "tactical_label", "prog_passes", "pressures", "tactical_intensity"]].head(5).iterrows():
    print(f" ⚽ {r['match']}:")
    print(f"    Estilo: {r['tactical_label']} (Intensidad: {int(r['tactical_intensity']*100)}%)")
    print(f"    Pases Progresivos: {r['prog_passes']} | Presiones Defensivas: {r['pressures']}")

--- VARIABLES NUMÉRICAS Y TÁCTICAS (JORNADA 29) ---
                                   match  xg_combined  table_parity  star_power  tactical_intensity  normalized_num_score
              Manchester City vs Arsenal         4.75          0.90       650.5                0.91                 92.79
            Tottenham Hotspur vs Chelsea         4.00          0.70       485.0                0.87                 72.12
     Newcastle United vs West Ham United         3.40          0.85       410.0                0.84                 66.81
  Aston Villa vs Wolverhampton Wanderers         3.25          0.75       415.0                0.83                 62.12
                    Liverpool vs Everton         3.80          0.30       470.0                0.86                 57.19
             Manchester United vs Fulham         2.85          0.70       390.0                0.81                 54.91
Brighton & Hove Albion vs Crystal Palace         2.95          0.70       335.0               

---
## 2. Entrenamiento 1: Calibración y Optimización de Pesos por Validación Cruzada ($k$-Fold CV)

El modelo canónico de recomendación pondera 4 dimensiones fundamentales:
$$Score_{partido} = w_1 \cdot xG_{comb} + w_2 \cdot Paridad + w_3 \cdot Fan\_Excitement + w_4 \cdot Media\_Velocity$$

Sujeto a las restricciones:
$$\sum_{i=1}^4 w_i = 1.0 \quad \text{y} \quad w_i \ge 0.05$$

Para encontrar los pesos óptimos sin incurrir en sobreajuste (*overfitting*), ejecutamos una búsqueda en simplex con **5-Fold Cross Validation** maximizando:
- Correlación de ranking de Spearman (preserva el orden relativo de atractivo de los partidos).
- Normalized Discounted Cumulative Gain (NDCG@3) para garantizar la calidad del Top 3 recomendado.
- Coeficiente de determinación $R^2$ y mínimo Mean Squared Error (MSE).

In [3]:
# 1. Instanciación y generación del benchmark sintético representativo (45 partidos)
optimizer = CrossValidationWeightOptimizer(k_folds=5, random_state=42)
X_bench, y_bench = optimizer._generate_synthetic_benchmark()

print(f"Dataset de Benchmark Histórico: {X_bench.shape[0]} partidos representativos con {X_bench.shape[1]} dimensiones")
print(f"Target Excitement Score ground-truth (Media: {np.mean(y_bench):.1f}, Min: {np.min(y_bench):.1f}, Max: {np.max(y_bench):.1f})")

# 2. Ejecución de la Búsqueda en Simplex con 5-Fold Cross Validation
cv_result = optimizer.optimize_weights_cv(X_bench, y_bench)
opt_w = cv_result.optimal_weights

print("\n--- RESULTADOS DE VALIDACIÓN CRUZADA (k=5 FOLDS) ---")
print(f" • Spearman Rank Correlation: {cv_result.spearman_rank_correlation:.3f}")
print(f" • NDCG@3 (Calidad del Ranking Top 3): {cv_result.ndcg_at_3:.3f}")
print(f" • Mean Squared Error (MSE): {cv_result.mean_squared_error:.3f}")
print(f" • Coeficiente R2: {cv_result.r2_score:.3f}")
print(f" • Estado: {cv_result.status.upper()}")

print("\n⚖️  PESOS ÓPTIMOS CALIBRADOS (PRESET 'cv_optimal'):")
print(f"   w_xG (Potencial Ofensivo)      = {opt_w.w_xg:.2f}")
print(f"   w_parity (Paridad en Tabla)    = {opt_w.w_parity:.2f}")
print(f"   w_fan (Sentimiento y Hype)     = {opt_w.w_fan_excitement:.2f}")
print(f"   w_media (Velocidad de Prensa)  = {opt_w.w_media_velocity:.2f}")
print(f"   Suma de pesos: {opt_w.w_xg + opt_w.w_parity + opt_w.w_fan_excitement + opt_w.w_media_velocity:.2f}")

Dataset de Benchmark Histórico: 45 partidos representativos con 4 dimensiones
Target Excitement Score ground-truth (Media: 65.7, Min: 33.2, Max: 95.1)

--- RESULTADOS DE VALIDACIÓN CRUZADA (k=5 FOLDS) ---
 • Spearman Rank Correlation: 0.963
 • NDCG@3 (Calidad del Ranking Top 3): 0.998
 • Mean Squared Error (MSE): 12.676
 • Coeficiente R2: 0.946
 • Estado: VALIDATED

⚖️  PESOS ÓPTIMOS CALIBRADOS (PRESET 'cv_optimal'):
   w_xG (Potencial Ofensivo)      = 0.20
   w_parity (Paridad en Tabla)    = 0.20
   w_fan (Sentimiento y Hype)     = 0.40
   w_media (Velocidad de Prensa)  = 0.20
   Suma de pesos: 1.00

---
## 3. Entrenamiento 2: Modelo No Lineal de Machine Learning Multimodal (`GradientBoostingRegressor`)

A diferencia de la formulación canónica lineal, partidos históricos muestran que ciertas interacciones cualitativo-numéricas no son aditivas:
- Un derbi de alta rivalidad histórica con alta intensidad de choque táctico produce dramatismo exponencial independientemente de la diferencia en la tabla.
- Un choque directo por el título entre dos clubes con altísimo Star Power genera interés desproporcionado.

Para capturar estas interacciones complejas, entrenamos un modelo supervisado **`GradientBoostingRegressor`** sobre **12 características multimodales**:
1. `norm_xg`: Potencial goleador normalizado
2. `norm_parity`: Paridad en tabla
3. `norm_star`: Star power de figuras clave
4. `tactical_intensity`: Choque táctico operativo
5. `fan_excitement_index`: Hype y sentimiento de afición
6. `media_velocity`: Velocidad de cobertura mediática
7. `narrative_rivalry_score`: Intensidad de derbi (TF-IDF)
8. `dramatic_intensity_score`: Probabilidad de desenlace dramático
9. `title_race_affinity`: Afinidad con lucha por el título
10. `derby_affinity`: Afinidad temática con derbi
11. `relegation_affinity`: Afinidad temática con descenso
12. `attacking_chaos_affinity`: Afinidad con fútbol dinámico y abierto

In [4]:
ml_scorer = HybridMLScorer(random_state=42)
gb_model = ml_scorer.model

print("--- MODELO GRADIENT BOOSTING ENTRENADO ---")
print(f" • Algoritmo: {type(gb_model).__name__}")
print(f" • Hiperparámetros:")
print(f"    - n_estimators = {gb_model.n_estimators}")
print(f"    - learning_rate = {gb_model.learning_rate}")
print(f"    - max_depth = {gb_model.max_depth}")
print(f"    - subsample = {gb_model.subsample}")
print(f" • Estado: {'Entrenado y Listo' if ml_scorer.is_trained else 'Pendiente'}")

--- MODELO GRADIENT BOOSTING ENTRENADO ---
 • Algoritmo: GradientBoostingRegressor
 • Hiperparámetros:
    - n_estimators = 50
    - learning_rate = 0.08
    - max_depth = 3
    - subsample = 0.85
 • Estado: Entrenado y Listo

### Interpretabilidad del Modelo: Feature Importances

Evaluamos la contribución porcentual relativa que cada una de las 12 características tiene en las decisiones de predicción del ensamble de árboles de Gradient Boosting.

In [5]:
telemetry = ml_scorer.get_telemetry()
feat_importances = telemetry.feature_importances

df_imp = pd.DataFrame([
    {
        "Variable": item.feature,
        "Importancia (%)": item.importance_percentage,
        "Descripción": item.description
    }
    for item in feat_importances
])

print("--- IMPORTANCIA RELATIVA DE CARACTERÍSTICAS (FEATURE IMPORTANCE) ---")
print(df_imp.to_string(index=False))

--- IMPORTANCIA RELATIVA DE CARACTERÍSTICAS (FEATURE IMPORTANCE) ---
                Variable  Importancia (%)                                                                 Descripción
                 norm_xg            57.07               Potencial goleador proyectado de ambos equipos (xG combinado)
      tactical_intensity            12.28                    Choque táctico operativo (pases progresivos y presiones)
          derby_affinity             5.79                                        Relevancia temática de clásico local
dramatic_intensity_score             5.60 Probabilidad de dramatismo y finales agónicos (clasificador de comentarios)
 narrative_rivalry_score             5.52                          Intensidad de derbi o rivalidad histórica (TF-IDF)
    fan_excitement_index             4.49          Nivel de expectativa y sentimiento de la afición en Reddit (VADER)
     relegation_affinity             3.83                           Relevancia temática de batalla por la

### Inferencia Multimodal y Atribución en la Jornada 29

Calculamos las predicciones del modelo `GradientBoostingRegressor` para cada partido de la jornada 29 y analizamos la descomposición de la puntuación en las 4 macro-dimensiones del espectáculo deportivo.

In [6]:
nlp_extractor = NLPFeatureExtractor()

ml_predictions = []
for f in fixtures:
    narrative = local_provider.get_narratives(f.id)
    num_feat = num_extractor.extract(f)
    nlp_feat = nlp_extractor.extract(f, narrative)
    
    score, attributions = ml_scorer.predict_score(num_feat, nlp_feat)
    
    ml_predictions.append({
        "fixture_id": f.id,
        "match": f"{f.home_team.name} vs {f.away_team.name}",
        "ml_excitement_score": score,
        "xg_dim_%": attributions.get("xg_offensive_potential", 0.0),
        "parity_dim_%": attributions.get("competitive_stakes_parity", 0.0),
        "fan_dim_%": attributions.get("fan_hype_and_drama", 0.0),
        "media_dim_%": attributions.get("media_coverage_and_stars", 0.0)
    })

df_ml = pd.DataFrame(ml_predictions).sort_values("ml_excitement_score", ascending=False).reset_index(drop=True)
print("--- PREDICCIONES DEL MODELO DE MACHINE LEARNING Y ATRIBUCIÓN PORCENTUAL ---")
print(df_ml[["match", "ml_excitement_score", "xg_dim_%", "parity_dim_%", "fan_dim_%", "media_dim_%"]].to_string(index=False))

print("\n✅ Fase 3 completada: Modelos calibrados, optimizados y evaluados con telemetría completa.")

--- PREDICCIONES DEL MODELO DE MACHINE LEARNING Y ATRIBUCIÓN PORCENTUAL ---
                                   match  ml_excitement_score  xg_dim_%  parity_dim_%  fan_dim_%  media_dim_%
              Manchester City vs Arsenal                89.01      18.6          27.7       17.8         35.8
            Tottenham Hotspur vs Chelsea                69.91      20.7          21.8       26.7         30.8
  Aston Villa vs Wolverhampton Wanderers                68.17      12.1          28.1       37.1         22.7
                    Liverpool vs Everton                66.98      17.7          11.6       31.9         38.8
Brighton & Hove Albion vs Crystal Palace                65.13      13.9          30.4       36.1         19.7
             Manchester United vs Fulham                58.97      17.2          30.5       20.9         31.5
     Newcastle United vs West Ham United                56.03      17.5          36.4       20.5         25.6
     AFC Bournemouth vs Sheffield United    